# Goal

Сквозное суперисследование, шаги 1-4. Тут возвращаем логику работы от обзов на оптимизации, как это было в `18n_ppo_tr_frostbite_07_old`: т.е. `VisionHead` заморожен, но всё равно идём от обзов, чтобы был регуляризационный эффект от AMP. Также на 3-ем и 4-ом шагах ослабляем давление `prediction_loss`.

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07.ipynb'

# Curriculum

## set_common_hyperparameters

In [2]:
# @launchit.collect
def set_common_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    HP.general.is_torch_amp_on_rollout = False
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.vision_head.parent = dict(model='18d_world_model_09:183', weights='18d_world_model_09:183')
    HP.vision_head.is_trainable = False
    
    HP.encoder.parent = dict(model='18d_world_model_09:183', weights='18d_world_model_09:183')
    HP.encoder.is_trainable = True

    HP.agent.parent = None 
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = None
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.is_obs_usage_enforced = True # ИДЁМ ОТ ОБЗОВ В ПРИНУДИТЕЛЬНОМ ПОРЯДКЕ, НЕСМОТРЯ НА ЗАМОРОЖЕННЫЙ VisionHead

    HP.ppo.ob_features_dropout = 0
    HP.ppo.ob_features_gauss_noise = 0
    
    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.05)')
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    
    return HP

## Step 1

In [3]:
STEP1_LAUNCHES_COUNT = 18

In [4]:
# @launchit.collect


def set_hyperparameters(HP, optuna_study, optuna_trial):
    HP = set_common_hyperparameters(HP, optuna_study, optuna_trial)

    # НА ЭТОМ ШАГЕ ИСПОЛЬЗУЕМ ОДНОГО РОДИТЕЛЯ!!! - см. set_common_hyperparameters
    
    # Test params
    HP.test.env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.test.env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # *
    ]
    HP.test.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_right_at_the_igloo_door'], # 0 
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_very_near_igloo_door'], # 1
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_near_center'], # 2
        ['no_score', 'last_life', 'one_remaining_igloo',   'temperature_10', 'bailey_near_center'], # 3
        ['no_score', 'last_life', 'three_remaining_igloo', 'temperature_20', 'bailey_near_center', ], # 4
        ['no_score', 'last_life', 'half_igloo', 'bailey_near_center'], # 5
        ['no_score', 'last_life', 'half_igloo'], # 6

        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'], # 7
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'], # 8
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'], # 9
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'], # 10
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # 11
    ]
    HP.ppo.rollout_env_stories = [
        '0;0:7',  # level 1 - teach agent to enter complete igloo
        '1;7:12', # level 5 - teach agent to enter complete blinking igloo
    ]
    
    return HP

In [5]:
# @launchit.collect
# @launchit.collect_step1
def set_hyperparameters(HP, optuna_study, optuna_trial):
    HP = set_common_hyperparameters(HP, optuna_study, optuna_trial)

    # НА ЭТОМ ШАГЕ ИСПОЛЬЗУЕМ ОДНОГО РОДИТЕЛЯ!!! - см. set_common_hyperparameters
    
    # Test params
    HP.test.env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.test.env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # *
    ]
    HP.test.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_right_at_the_igloo_door'], # 0 
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_very_near_igloo_door'], # 1
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_near_center'], # 2
        ['no_score', 'last_life', 'one_remaining_igloo',   'temperature_10', 'bailey_near_center'], # 3
        ['no_score', 'last_life', 'three_remaining_igloo', 'temperature_20', 'bailey_near_center', ], # 4
        ['no_score', 'last_life', 'half_igloo', 'bailey_near_center'], # 5
        ['no_score', 'last_life', 'half_igloo'], # 6

        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'], # 7
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'], # 8
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'], # 9
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'], # 10
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # 11
    ]
    HP.ppo.rollout_env_stories = [
        '0;0:7',  # level 1 - teach agent to enter complete igloo
        '1;7:12', # level 5 - teach agent to enter complete blinking igloo
    ]
    
    return HP

## Step 2

In [6]:
STEP2_LAUNCHES_COUNT = 18

In [7]:
# @launchit.collect



In [8]:
# @launchit.disable
# @launchit.collect_step2
# def set_hyperparameters(HP, optuna_study, optuna_trial):
#     HP = set_common_hyperparameters(HP, optuna_study, optuna_trial)
# 
#     parent = optuna_trial.suggest_categorical('parent', ''.split(','))
#     HP.encoder.parent['weights'] = parent
#     HP.agent.parent = parent
# 
#     HP.ppo.rollout_env_rams = [
#         'com.develorium.neurolab.frostbite_ram:level1:1:cls=train1',
#         'com.develorium.neurolab.frostbite_ram:level1:1:cls=train2',
#         'com.develorium.neurolab.frostbite_ram:level1:1:cls=train3',
#         'com.develorium.neurolab.frostbite_ram:level1:1:cls=train4',
#         'com.develorium.neurolab.frostbite_ram:level1:1:cls=train5',
#     ]
#     HP.ppo.rollout_env_ram_patches = [
#         ['no_score', 'last_life', 'no_igloo', 'temperature_45', 'bailey_safe_random_spawn'],
#     ]
# 
#     return HP

## Step 3

In [9]:
STEP3_LAUNCHES_COUNT = 18

In [10]:
# @launchit.collect



In [11]:
# @launchit.disable
# @launchit.collect_step3
# def set_hyperparameters(HP, optuna_study, optuna_trial):
#     HP = set_common_hyperparameters(HP, optuna_study, optuna_trial)
# 
#     parent = optuna_trial.suggest_categorical('parent', ''.split(','))
#     HP.encoder.parent['weights'] = parent
#     HP.agent.parent = parent
# 
#     HP.ppo.rollout_env_ram_patches = [
#         ['nine_lives'], 
#     ]
#     HP.ppo.prediction_loss = dict(coef=0.02) # УМЕНЬШАЕМ ДАВЛЕНИЕ PREDICTION_LOSS - СТИМУЛИРУЕМ EXPLORATION
#     
#     return HP

## Step 4

In [12]:
STEP4_LAUNCHES_COUNT = 18

In [13]:
# @launchit.collect



In [14]:
# @launchit.disable
# @launchit.collect_step4
# def set_hyperparameters(HP, optuna_study, optuna_trial):
#     HP = set_common_hyperparameters(HP, optuna_study, optuna_trial)
#     
#     parent = optuna_trial.suggest_categorical('parent', ''.split(','))
#     HP.encoder.parent['weights'] = parent
#     HP.agent.parent = parent
# 
#     HP.ppo.rollout_env_ram_patches = [
#         ['nine_lives'], 
#     ]
# 
#     HP.ppo.prediction_loss = dict(coef=0.02) # УМЕНЬШАЕМ ДАВЛЕНИЕ PREDICTION_LOSS - СТИМУЛИРУЕМ EXPLORATION
#     HP.ppo.entropy_loss = dict(coef='const(0.03)') # КРИСТАЛИЗУЕМ УСПЕШНОЕ ПОВЕДЕНИЕ
#     
#     return HP

# Results


# System

In [15]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple
from enum import StrEnum, auto

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = '/home/misha/dev/mine/neurolab'
# @launchit.disable
# project_root_path = ! git rev-parse --show-toplevel
# project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [16]:
class ExecMode(StrEnum):
    SUPERSTUDY = auto()
    STUDY = auto()

CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, studies_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'exec_mode, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    studies_path=os.path.join(os.path.abspath('.'), 'studies'),
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    exec_mode=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = lu.coalesce(json.load(connection_file).get('jupyter_session'), '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_35/studies/18n_study_35.1.ipynb')
    assert os.path.exists(optuna_study_notebook_fname)
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    exec_mode = lu.when('superstudy' in optuna_study_name, ExecMode.SUPERSTUDY, ExecMode.STUDY)
    CONFIG = CONFIG._replace(exec_mode=exec_mode)
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'studies_path': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_35/studies',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07',
 'exec_mode': <ExecMode.STUDY: 'study'>,
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_35/studies/18n_study_35.1.ipynb',
 'optuna_study_name': '18n_study_35.1',
 'optuna_study_serial': '35.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_35/studies/18n_study_35.1.optuna'}

In [17]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)
os.makedirs(CONFIG.run_path, exist_ok=True)
os.makedirs(CONFIG.studies_path, exist_ok=True)

In [18]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

# Superstudy mode

## discover_step_numbers

In [19]:
def discover_step_numbers():
    nbp = launchit.NotebookProcessor()
    
    with open(CONFIG.optuna_study_notebook_fname, 'rt') as f:
        nbp(f, 'notebook.ipynb', expandvars={}, collect_inds=[], disable_inds=[])

    step_nos = []

    for collect_ind in filter(lambda x: x is not None, nbp.found_collect_inds):
        m = re.match(r'step(\d+)', collect_ind)
        
        if m:
            step_nos.append(int(m.group(1)))

    return sorted(step_nos)

## create_optuna_study

In [20]:
def create_optuna_study(step_no, parents):
    study_name = f'{CONFIG.optuna_study_name.replace('superstudy', 'study')}.{step_no}'
    study_notebook_fname = os.path.join(CONFIG.studies_path, f'{study_name}.ipynb')
    is_created = False

    if not os.path.exists(study_notebook_fname):
        expandvars = dict(
            PROJECT_ROOT_PATH=CONFIG.project_root_path,
            OPTUNA_STUDY_NOTEBOOK_FNAME=study_notebook_fname,
            PARENTS=parents,
            STEP_NO=str(step_no),
        )
        launchit.launchit(
            CONFIG.optuna_study_notebook_fname, 
            expandvars=expandvars, 
            make_py_file=False, 
            dir_name=CONFIG.run_path,
            collect_inds=[f'step{step_no}'],
            disable_inds=[],
            new_fname=study_notebook_fname,
        )
        is_created = True
        
    return study_notebook_fname, study_name, is_created

## Unleash!

In [21]:
if CONFIG.exec_mode == ExecMode.SUPERSTUDY:
    step_nos = discover_step_numbers()
    LOG(f'Step numbers: {step_nos}')
    top_parents = []

    for step_no in step_nos:
        step_study_notebook_fname, step_study_name, is_step_study_created = create_optuna_study(step_no, parents=','.join(top_parents))

        if is_step_study_created:
            LOG(f'Launching step study "{step_study_notebook_fname}" for parents={top_parents}')
            subprocess.run(
                ['papermill', step_study_notebook_fname, step_study_notebook_fname, '--no-progress-bar'],
                capture_output=False,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                check=True,
            )
        else:
            LOG(f'Step study "{step_study_notebook_fname}" is already done')

        step_study = optuna.create_study(
            study_name=step_study_name,
            storage=JournalStorage(JournalFileBackend(file_path=re.sub('.ipynb$', '.optuna', step_study_notebook_fname))),
            load_if_exists=True, 
        )

        top_trials = sorted(filter(lambda t: t.values, step_study.trials), key=lambda t: -t.values[0])[:3]
        LOG(f'Top trials for step {step_no}:')

        for i, trial in enumerate(top_trials):
            LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

        top_parents = list(map(lambda t: CONFIG.target_notebook_name + ':' + t.user_attrs['MODEL_VERSION'], top_trials))

# Study mode

## create_optuna_launch

In [22]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

## run_optuna_launch

In [23]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [24]:
if CONFIG.exec_mode == ExecMode.STUDY:
    optuna_study = optuna.create_study(
        study_name=CONFIG.optuna_study_name,
        directions=['maximize'],
        storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
        load_if_exists=True,
    )
    optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
    launches_count = globals()['STEP' + '1' + '_LAUNCHES_COUNT']
    completed_launches_count = 0
    
    with LOG.auto_log_level(logging.INFO):
        with cf.ThreadPoolExecutor(max_workers=32) as executor:
            futures = {}
            idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
            is_first_time = True
            
            while launches_count is None or completed_launches_count < launches_count:
                runners_info = launch_dispatcher.RunnersInfo.get()
                idle_runners_af(runners_info['idle'])
    
                if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                    if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                        launch_name, launch_fname = create_optuna_launch()
                        futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                        LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                        idle_runners_af.reset()
                        
                    is_first_time = False
    
                try:
                    while futures:
                        completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)
    
                        if not completed_futures:
                            break
                            
                        for completed_future in completed_futures:
                            launch_name = futures[completed_future]
                            del futures[completed_future]
    
                            exc = completed_future.exception()
                            
                            if exc is not None:
                                LOG(f'Launch "{launch_name}" failed: {exc}')
                            else:
                                LOG(f'Launch "{launch_name}" completed')
        
                        if completed_futures:
                            completed_launches_count += len(completed_futures)
                            LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
                except TimeoutError as e:
                    pass
    
                time.sleep(5)

[I 2026-10-08 23:42:45,295] A new study created in Journal with name: 18n_study_35.1


2026.10.08-23:42:45.707273     0.378 >> Model instance registered, version=944


2026.10.08-23:42:45.726958     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch944.ipynb"


2026.10.08-23:42:45.728114     0.007 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:944"; running launches=1


2026.10.08-23:43:16.794198     0.330 >> Model instance registered, version=945


...

2026.10.09-02:42:56.944871     0.002 >> Launch "18n_ppo_tr_frostbite_07:959" completed


2026.10.09-02:42:56.945953     0.001 >> 17 (+1) launches completed; running launches=1


2026.10.09-02:43:17.159283     4.872 >> Trial 17 (18n_ppo_tr_frostbite_07:961) finished with value: 0.7 and parameters: {}. Best is trial 12 with value: 0.896875


2026.10.09-02:43:17.405751     0.002 >> Launch "18n_ppo_tr_frostbite_07:961" completed


2026.10.09-02:43:17.406757     0.001 >> 18 (+1) launches completed; running launches=0


In [25]:
if CONFIG.exec_mode == ExecMode.STUDY:
    study = optuna.create_study(
        study_name=optuna_study_name,
        storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
        load_if_exists=True, 
    )
    
    pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])
    
    LOG('Study statistics: ')
    LOG(f'\tNumber of finished trials: {len(study.trials)}')
    LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
    LOG(f'\tNumber of complete trials: {len(complete_trials)}')
    
    if len(study.directions) == 1:
        LOG('Best trial:')
        trial = study.best_trial
        
        LOG(f'\tValue: {trial.value}')
        LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
        
        LOG('\tParams: ')
        
        for key, value in trial.params.items():
            LOG(f'\t\t{key}: {value}')
    else:
        LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")
    
        for i in range(3):
            LOG(f"Trial with lowest loss_{i}:")
            trial = min(study.best_trials, key=lambda t: t.values[i])
            LOG(f"\tnumber: {trial.number}")
            LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
            LOG(f"\tparams: {trial.params}")
            LOG(f"\tvalues: {trial.values}")

[I 2026-10-09 02:43:22,445] Using an existing study with name '18n_study_35.1' instead of creating a new one.


2026.10.09-02:43:22.446034     5.039 >> Study statistics: 


2026.10.09-02:43:22.447241     0.001 >> 	Number of finished trials: 18


2026.10.09-02:43:22.447732     0.000 >> 	Number of pruned trials: 0


2026.10.09-02:43:22.448177     0.000 >> 	Number of complete trials: 18


2026.10.09-02:43:22.448743     0.001 >> Best trial:


2026.10.09-02:43:22.449156     0.000 >> 	Value: 0.896875


2026.10.09-02:43:22.449336     0.000 >> 	Model version: 956


2026.10.09-02:43:22.449504     0.000 >> 	Params: 
